# Regresión Logística con PyTorch — Detección de Ataques en Redes IoT

**Dataset:** CICIoT2023 (xsmall)  
**Fuente:** https://www.kaggle.com/datasets/madhavmalhotra/unb-cic-iot-dataset

## Variable objetivo (Y)
- **0** = Benigno (tráfico normal)
- **1** = Malicioso (cualquier tipo de ataque)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from matplotlib import pyplot
from sklearn.metrics import accuracy_score

%matplotlib inline

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Carga y Preprocesamiento con Pandas

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Intelegencia Artificial/Dataset/CICIoT2023_xxsmall.csv')

print(f'Dimensiones: {data.shape}')
print(data['label'].value_counts())

In [ ]:
conteo_clases = data['label'].value_counts()

pyplot.figure(figsize=(12, 5))

pyplot.subplot(1, 2, 1)
conteo_clases.plot(kind='bar', color='steelblue', ax=pyplot.gca())
pyplot.title('Distribución de Tipos de Tráfico\n(categorías originales)')
pyplot.xlabel('Tipo de tráfico')
pyplot.ylabel('Cantidad de registros')
pyplot.xticks(rotation=45, ha='right', fontsize=7)
pyplot.grid(axis='y', alpha=0.4)

pyplot.subplot(1, 2, 2)
benigno   = (data['label'].str.contains('Benign', case=False)).sum()
malicioso = len(data) - benigno
pyplot.bar(['Benigno', 'Malicioso'], [benigno, malicioso], color=['green', 'red'])
pyplot.title('Benigno vs Malicioso')

pyplot.tight_layout()
pyplot.show()

In [ ]:
data['label_bin'] = data['label'].apply(lambda x: 0 if 'Benign' in str(x) else 1)

FEATURES = [
    'flow_duration', 'Header_Length', 'Protocol Type', 'Duration',
    'Rate', 'Srate', 'Drate',
    'fin_flag_number', 'syn_flag_number', 'rst_flag_number',
    'psh_flag_number', 'ack_flag_number', 'ece_flag_number', 'cwr_flag_number',
    'HTTP', 'HTTPS', 'DNS', 'Telnet', 'SMTP', 'SSH', 'IRC',
    'TCP', 'UDP', 'DHCP', 'ARP', 'ICMP', 'IPv', 'LLC',
    'Tot sum', 'Min', 'Max', 'AVG', 'Std', 'Tot size', 'IAT',
    'Number', 'Magnitue', 'Radius', 'Covariance', 'Variance', 'Weight'
]
FEATURES = [f for f in FEATURES if f in data.columns]
TARGET   = 'label_bin'

for col in FEATURES:
    if data[col].isnull().sum() > 0:
        data[col] = data[col].fillna(data[col].median())

X = data[FEATURES].values.astype(np.float32)
y = data[TARGET].values.astype(np.float32)
m = y.size

print(f'X shape : {X.shape}')
print(f'y shape : {y.shape}')

In [ ]:
def plotData(X, y):
    fig = pyplot.figure()
    pos = y == 1
    neg = y == 0
    pyplot.plot(X[pos, 0], X[pos, 1], 'k*', lw=2, ms=10)
    pyplot.plot(X[neg, 0], X[neg, 1], 'ko', mfc='y', ms=8, mec='k', mew=1)

plotData(X, y)
pyplot.xlabel('Duración del Flujo (microsegundos)')
pyplot.ylabel('Longitud del Encabezado (bytes)')
pyplot.legend(['Malicioso', 'Benigno'])
pass

## 2. Normalización y Split 80/20

In [ ]:
# se calcula la media y desviacion estandar para normalizar los datos
mu    = np.mean(X, axis=0)
sigma = np.std(X, axis=0)
sigma[sigma == 0] = 1
X = ((X - mu) / sigma).astype(np.float32)

np.random.seed(42)
indices = np.random.permutation(m)
split   = int(0.8 * m)

X_train = X[indices[:split]]
y_train = y[indices[:split]]
X_test  = X[indices[split:]]
y_test  = y[indices[split:]]

print(f'Entrenamiento : {len(y_train):,}')
print(f'Validación    : {len(y_test):,}')

In [ ]:
# se convierten los arrays de numpy a tensores de pytorch
X_t = torch.from_numpy(X_train).float()
Y_t = torch.from_numpy(y_train).float().view(-1, 1)

print(f'X_t shape: {X_t.shape}')
print(f'Y_t shape: {Y_t.shape}')

## 3. Definir el Modelo

Se crea una clase que hereda de `torch.nn.Module`.
En `__init__` se definen las capas y en `forward` la lógica de cálculo.

In [ ]:
# creamos una clase que hereda de torch.nn.Module
class ModeloRegresionLogistica(nn.Module):

    # constructor
    def __init__(self, D_in, D_out):

        # llamamos al constructor de la clase madre
        super(ModeloRegresionLogistica, self).__init__()

        # definimos la capa lineal
        self.fc = nn.Linear(D_in, D_out)

    # lógica para calcular las salidas de la red
    def forward(self, x):
        x = torch.sigmoid(self.fc(x))  # Función de activación sigmoidal
        return x


D_in  = X_t.shape[1]  # numero de features
D_out = 1             # binario: 0 o 1

model = ModeloRegresionLogistica(D_in, D_out)
print(model)

# verificar que el modelo recibe los datos en la forma correcta
x_prueba = torch.randn(5, D_in)
outputs  = model(x_prueba)
print(outputs.shape)

## 4. Función de Pérdida y Optimizador

In [ ]:
criterion = nn.BCELoss()  # Binary Cross Entropy para clasificacion binaria
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)  # Descenso de gradiente estocástico

## 5. Entrenamiento

In [ ]:
epochs   = 500
log_each = 100
l = []
model.train()
for e in range(1, epochs + 1):

    # se calculan las predicciones con los parametros actuales
    y_pred = model(X_t)

    # se calcula el error entre la prediccion y el valor real
    loss = criterion(y_pred, Y_t)
    l.append(loss.item())

    # se limpian los gradientes del paso anterior para no acumularlos
    optimizer.zero_grad()

    # se calculan automaticamente todos los gradientes
    loss.backward()

    # se actualizan los parametros del modelo con los gradientes calculados
    optimizer.step()

    if not e % log_each:
        print(f'Epoch {e}/{epochs} Loss {np.mean(l):.5f}')

## 6. Gráfica de Convergencia del Costo

In [ ]:
pyplot.plot(np.arange(len(l)), l, lw=2)
pyplot.xlabel('Número de iteraciones')
pyplot.ylabel('Costo J')
pass

## 7. Evaluación del Modelo

In [ ]:
def evaluate(x):
    model.eval()
    with torch.no_grad():
        probs = model(x).squeeze().numpy()
        return (probs >= 0.5).astype(int), probs

pred_train, _ = evaluate(X_t)
print('Precisión entrenamiento: {:.2f}%'.format(
    accuracy_score(y_train, pred_train) * 100))

pred_test, prob_test = evaluate(torch.from_numpy(X_test).float())
print('Precisión validación   : {:.2f}%'.format(
    accuracy_score(y_test, pred_test) * 100))

## 8. Predicciones finales

In [ ]:
print('Precisión del conjunto de validación: {:.2f}%'.format(
    accuracy_score(y_test, pred_test) * 100))
print()
print('Predicciones finales:')
for i in range(20):
    real     = int(y_test[i])
    pred     = int(pred_test[i])
    prob     = prob_test[i]
    correcto = '✓' if real == pred else '✗'
    etiq_real = 'Malicioso' if real == 1 else 'Benigno  '
    etiq_pred = 'Malicioso' if pred == 1 else 'Benigno  '
    print(f'Real: {etiq_real}  Predicción: {etiq_pred}  Prob: {prob:.4f}  {correcto}')